In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

import planetary_computer as pc
from pystac_client import Client
from pystac.extensions.eo import EOExtension as eo

import rioxarray
import xarray as xr
from rioxarray.merge import merge_arrays
from shapely.geometry import box
from einops import rearrange
from PIL import Image
from tqdm.notebook import tqdm

from garuda.ops import local_to_geo

import pyproj

## Specify the region of interest

In [ ]:
ref_dir = "/mnt/dataset/brick_kilns/processed_data/lucknow_sarath_grid_obb_v3/images"
images = glob(ref_dir + "/*.png")
lats = [float(img.split("/")[-1].replace(".png","").split(",")[0]) for img in images]
lons = [float(img.split("/")[-1].replace(".png","").split(",")[1]) for img in images]
max_lat = max(lats)+0.02
min_lat = min(lats)-0.02
max_lon = max(lons)+0.02
min_lon = min(lons)-0.02
print(max_lat, min_lat, max_lon, min_lon)

## Download cloudless imagery

In [ ]:
polygon = box(min_lon, min_lat, max_lon, max_lat)
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=pc.sign_inplace)
time_of_interest = "2024-03-08/2024-03-08"

search = catalog.search(
        collections=["sentinel-2-l2a"],
        intersects=polygon,
        datetime=time_of_interest,
        query={"eo:cloud_cover": {"lt": 1}},
    )
items = search.item_collection()
print(f"Returned {len(items)} Items")

sorted_items = sorted(items, key=lambda item: eo.ext(item).cloud_cover)
# print("cloud covers: ", [eo.ext(item).cloud_cover for item in sorted_items])
print(type(sorted_items))

for i, item in enumerate(sorted_items):
    print(item.properties['datetime'], eo.ext(item).cloud_cover)
    raster_image = rioxarray.open_rasterio(pc.sign(item.assets["visual"].href))
    transform = pyproj.Transformer.from_crs(raster_image.rio.crs, "EPSG:4326")

    # print("raster_image1: ", raster_image.rio.bounds())
    # print("raster_image2: ", raster_image2.rio.bounds())

    r1_lat_min, r1_lon_min = transform.transform(raster_image.rio.bounds()[0], raster_image.rio.bounds()[1])
    r1_lat_max, r1_lon_max = transform.transform(raster_image.rio.bounds()[2], raster_image.rio.bounds()[3])
    print("r1_lat_min, r1_lon_min, r1_lat_max, r1_lon_max: ", r1_lat_min, r1_lon_min, r1_lat_max, r1_lon_max)
    
    plt.plot([r1_lon_min, r1_lon_max], [r1_lat_min, r1_lat_min], color=f'C{i}')
    plt.plot([r1_lon_min, r1_lon_max], [r1_lat_max, r1_lat_max], color=f'C{i}')
    plt.plot([r1_lon_min, r1_lon_min], [r1_lat_min, r1_lat_max], color=f'C{i}')
    plt.plot([r1_lon_max, r1_lon_max], [r1_lat_min, r1_lat_max], color=f'C{i}', label=f"raster_image_{i}")
    
plt.plot([min_lon, max_lon], [min_lat, min_lat], color='yellow')
plt.plot([min_lon, max_lon], [max_lat, max_lat], color='yellow')
plt.plot([min_lon, min_lon], [min_lat, max_lat], color='yellow')
plt.plot([max_lon, max_lon], [min_lat, max_lat], color='yellow', label="area of interest")

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

## Merge four tiles into one (may take a while)

In [ ]:
img1 = rioxarray.open_rasterio(pc.sign(sorted_items[0].assets["visual"].href))
img2 = rioxarray.open_rasterio(pc.sign(sorted_items[1].assets["visual"].href))
img3 = rioxarray.open_rasterio(pc.sign(sorted_items[2].assets["visual"].href))
img4 = rioxarray.open_rasterio(pc.sign(sorted_items[3].assets["visual"].href))
img = merge_arrays([img1, img2, img3, img4], method='first')

## Save the merged image as a GeoTIFF

In [ ]:
img.rio.to_raster(f"lucknow_sarath_grid_{time_of_interest.replace('/', '_')}.tif")

## Run notebook from here after saving

In [ ]:
loaded_img = rioxarray.open_rasterio(f"/home/username/kilns_neurips24/experiments/lucknow_sarath_grid_2024-03-08_2024-03-08.tif")
loaded_img

## Visualizations

In [ ]:
from einops import rearrange
values = loaded_img.values
values = rearrange(values, 'c y x -> y x c') / 255.0
values = values[::100, ::100]
print(values.shape)

In [ ]:
x = loaded_img.x.values[::100]
y = loaded_img.y.values[::100]

X, Y = np.meshgrid(x, y)

plt.pcolormesh(X, Y, values)

inverse_transform = pyproj.Transformer.from_crs("EPSG:4326", raster_image.rio.crs)
min_x, min_y = inverse_transform.transform(min_lat, min_lon)
plt.scatter(min_x, min_y, color='red', label="min_lat, min_lon")
max_x, max_y = inverse_transform.transform(max_lat, max_lon)
plt.scatter(max_x, max_y, color='green', label="max_lat, max_lon")
plt.legend()

## Export the imagery as PNG and transform the labels

In [ ]:
width = 128
height = 128
scale = 5
labels_dir = ref_dir.replace("images", "labels")
label_files = glob(labels_dir + "/*.txt")
print(len(label_files))

save_dir = f"/home/username/kilns_neurips24/processed_data/lucknow_sarath_grid_obb_v3_sentinel_v1_{width*scale}"
os.makedirs(f"{save_dir}/images", exist_ok=True)
os.makedirs(f"{save_dir}/labels", exist_ok=True)

for file in tqdm(label_files):
    base_name = os.path.basename(file)
    str_lat, str_lon = base_name.replace(".txt", "").split(",")
    img_center_lat, img_center_lon = float(str_lat), float(str_lon)
    x, y = inverse_transform.transform(str_lat, str_lon)
    
    # find closest index of x, y in loaded_img.x, loaded_img.y
    x_idx = np.abs(loaded_img.x - x).argmin().item()
    y_idx = np.abs(loaded_img.y - y).argmin().item()
    
    sel_box = loaded_img.isel(x=slice(x_idx-width//2, x_idx+width//2), y=slice(y_idx-height//2, y_idx+height//2))
    png_data = rearrange(sel_box.values, 'c y x -> y x c')
    
    # reproject the label
    label_items = np.loadtxt(file, ndmin=2)
    re_label_items = []
    for label in label_items:
        xyxyxyxy = label[1:9]
        xyxyxyxy = xyxyxyxy.reshape(4, 2)
        geo = local_to_geo(xyxyxyxy[:, 0], xyxyxyxy[:, 1], 17, img_center_lat, img_center_lon, 1120, 1120)
        utm_x, utm_y = inverse_transform.transform(geo[:, 0], geo[:, 1])
        
        sel_box_x = np.linspace(sel_box.x.min(), sel_box.x.max(), width*scale)
        sel_box_y = np.linspace(sel_box.y.max(), sel_box.y.min(), height*scale)
        
        x_idx = np.array([np.abs(sel_box_x - utm_x[i]).argmin().item() for i in range(4)])
        y_idx = np.array([np.abs(sel_box_y - utm_y[i]).argmin().item() for i in range(4)])
        # normalize
        x_idx = x_idx / width / scale
        y_idx = y_idx / height / scale
        re_label = np.array([label[0], x_idx[0], y_idx[0], x_idx[1], y_idx[1], x_idx[2], y_idx[2], x_idx[3], y_idx[3]])
        re_label_items.append(re_label)
    
    re_label = np.array(re_label_items)
    np.savetxt(f"{save_dir}/labels/{base_name}", re_label, fmt='%d %f %f %f %f %f %f %f %f')
    
    # save png file
    png_data = png_data.astype(np.uint8)
    png_img = Image.fromarray(png_data)
    png_img.save(f"{save_dir}/images/{base_name.replace('.txt', '.png')}")

print(f"Saved {len(label_files)} images and labels in {save_dir}")